# OpenAI Built-in Web Search Tool

In [11]:
import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace, WebSearchTool

dotenv.load_dotenv()

True

Let's set up our RAG database connection:

In [12]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [13]:
# This is the same code as in the rag.ipynb notebook


@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Use OpenAI's built-in WebSearchTool instead of external MCP:

In [ ]:
# OpenAI's built-in WebSearchTool - no API key or MCP connection needed
# This is a hosted tool managed by OpenAI that searches the web.

calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers. Never ask follow-up questions - always provide a complete answer.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, use web search to figure out the typical/standard ingredients and portions of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, use the calorie_lookup_tool to get the calorie information for each ingredient.
    * Even if you know the recipe of the meal, always use web search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * For meals, always provide a full itemized breakdown with:
        - Each ingredient with its typical portion size
        - Calories for that portion
        - Total calories at the end
    * Assume standard/typical portions when not specified. Do not ask the user for clarification.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
)

Reference query - shouldn't use web search:

In [16]:
with trace("Nutrition Assistant with Web Search - Only uses calorie_lookup_tool"):
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    - Calories for typical sizes (per piece):
      - Banana (medium, ~118 g): ~105 kcal
      - Apple (medium, ~182 g): ~95 kcal
      - Total for one banana + one apple: ~200 kcal
    
    - Calories per 100 g:
      - Banana: 89 kcal/100 g
      - Apple: 52 kcal/100 g
    
    Notes: Actual calories vary with size and variety.
- 7 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [18]:
with trace("Nutrition Assistant with Web Search"):
    result = await Runner.run(
        calorie_agent_with_search, 
        "How many calories are in an english breakfast?",
        max_turns=20  # Increase from default 10 - complex meals need more tool calls
    )
    print(result.final_output)

A traditional full English breakfast is roughly 800–1000 calories per serving, depending on portions (eggs, bacon, sausage, beans, tomatoes, mushrooms, toast, hash browns). If you want a itemized breakdown, tell me which items you include and typical portions.
